# 7. LangSmith (Tracking your GenAI app)

**LangSmith** is a website that **records and shows everything your AI app does**.

Every time your app runs, LangSmith saves:
- the question that came in,
- the prompt sent to the model,
- the model's reply,
- how long it took, and how many tokens (cost) it used.

This recording is called **tracing**. It helps you **see, debug, and improve** your app.

## Real-life analogy

LangSmith is like a **CCTV camera** for your AI app.

- Without it, you only see the final answer — you can't tell *why* the app did something.
- With it, you can **play back every step** and find exactly where things went wrong.

This is very useful when an answer looks strange and you need to know *why*.

## Step 1: Get a free LangSmith key

1. Go to **https://smith.langchain.com** and sign up (free).
2. Open **Settings → API Keys** and create a key.
3. Copy the key — you'll use it below.

## Step 2: Install

In the terminal:
```
pip install langsmith langchain langchain-openai
```

## Step 3: Connect by setting 4 environment variables

LangSmith turns on automatically when these are set. You can set them in PowerShell **before** starting Jupyter:
```
$env:LANGSMITH_TRACING = "true"
$env:LANGSMITH_API_KEY = "your-langsmith-key"
$env:LANGSMITH_PROJECT = "my-first-app"     # any name you like
$env:OPENAI_API_KEY    = "your-openai-key"
```

Or set them inside the notebook (just for testing):

In [ ]:
import os

# Turn LangSmith tracking ON
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "your_api_key"   # <-- paste your key
os.environ["LANGSMITH_PROJECT"] = "my-first-app"          # runs are grouped under this name
os.environ["OPENAI_API_KEY"] = "your_api_key"          # <-- paste your key

print("LangSmith is connected!")

LangSmith is connected!


| Variable | What it does |
|----------|--------------|
| `LANGSMITH_TRACING` | Set to `"true"` to switch tracking ON |
| `LANGSMITH_API_KEY` | Your LangSmith key (proves who you are) |
| `LANGSMITH_PROJECT` | A folder name to group your runs |
| `OPENAI_API_KEY` | Your OpenAI key (the model that runs) |

## Step 4: Run a LangChain chain — it is traced automatically

If you use LangChain (like the Chains notebook), **you don't write any extra code**. Once the variables above are set, every run is sent to LangSmith on its own.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template("Explain {topic} in one simple sentence.")
model = ChatOpenAI(model="gpt-4o-mini")
chain = prompt | model | StrOutputParser()

# This run is automatically recorded in LangSmith.
print(chain.invoke({"topic": "LangSmith"}))

LangSmith is a platform designed to streamline the development and management of language models by providing tools for collaboration, versioning, and deployment.


Now open **https://smith.langchain.com**, click your project **`my-first-app`**, and you'll see this run with the prompt, the reply, the time taken, and the token count.

## Step 5: Track your OWN functions with `@traceable`

What if you have your own Python function (not a LangChain chain)? Add `@traceable` on top of it, and it shows up as a step in LangSmith too.

In [ ]:
from langsmith import traceable

@traceable
def make_greeting(name):
    return f"Hello {name}, welcome to GenAI!"

# This call is now recorded in LangSmith as its own step.
print(make_greeting("Thiru"))

## Step 6: Track the raw OpenAI client with `wrap_openai`

If you call OpenAI directly (without LangChain), wrap the client once and every call is traced.

In [ ]:
from openai import OpenAI
from langsmith.wrappers import wrap_openai

# wrap_openai is the ONLY change needed for tracking.
client = wrap_openai(OpenAI())

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say hello in one word."}],
)

print(response.choices[0].message.content)

## What you can see in LangSmith

- **Traces** — every run, step by step, like a timeline.
- **Inputs & outputs** — the exact prompt and the exact reply.
- **Latency** — how many seconds each step took.
- **Tokens & cost** — how much each call used.
- **Errors** — which step failed and why.

## Key points to remember

- **LangSmith** records and shows everything your AI app does (this is **tracing**).
- It is like a **CCTV camera** for your app — great for debugging.
- Connect it by setting 4 env vars: `LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`, `OPENAI_API_KEY`.
- **LangChain chains are traced automatically** — no extra code.
- Use **`@traceable`** to track your own functions.
- Use **`wrap_openai`** to track direct OpenAI calls.
- View everything at **https://smith.langchain.com**.